## Imports

In [1]:
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori

print("Bibliotheques importees.")

Bibliotheques importees.


## Paramètres

In [2]:
OUTPUT_DIR = "../../outputs/concrete"
TARGET_COL = "Concrete compressive strength"

MIN_SUPPORT  = 0.25
MIN_LENGTH   = 1
MIN_SEQ_SIZE = 2

## Chargement des données d'entraînement UNIQUEMENT

In [3]:
print("[TG-Patterns] Chargement des donnees d'entrainement (issues du notebook 1)...")

X_train = pd.read_csv(f"{OUTPUT_DIR}/X_train.csv", sep=";")
y_train = pd.read_csv(f"{OUTPUT_DIR}/y_train.csv", sep=";")[TARGET_COL]

train_df = X_train.copy()
train_df[TARGET_COL] = y_train.values

print(f"[TG-Patterns] {train_df.shape[0]} lignes d'entrainement chargees.")

[TG-Patterns] Chargement des donnees d'entrainement (issues du notebook 1)...
[TG-Patterns] 824 lignes d'entrainement chargees.


## LA fonction unique de la Phase 1

In [4]:
def extraire_motifs_graduels(data_train, target_col, ascending,
                              min_support=0.25, min_length=1, min_seq_size=2):
    """
    Fonction UNIQUE de la Phase 1 (voir memoire, section 3.3).

    Entrees :
        data_train   : DataFrame des donnees d'ENTRAINEMENT (features + cible)
        target_col   : nom de la colonne cible
        ascending    : True  -> motifs associes a une HAUSSE de la cible
                       False -> motifs associes a une BAISSE de la cible
        min_support  : support minimal (sigma) pour Apriori
        min_length   : longueur minimale d'un motif
        min_seq_size : taille minimale d'une sequence consecutive

    Sortie :
        DataFrame des sequences consecutives, colonnes :
        motif, support, seq_num, taille, cible_depart, cible_arrivee
        (triees par cible_depart decroissant)
    """
    print(f"[Phase 1] Extraction des motifs graduels (ascending={ascending})...")

    print("[Phase 1] Etape 1/6 : tri du dataset selon la cible...")
    data_sorted = data_train.sort_values(by=target_col, ascending=ascending).reset_index(drop=True)

    print("[Phase 1] Etape 2/6 : transformation numToCat (objets consecutifs)...")
    data_features = data_sorted.drop(columns=[target_col])
    data_target   = data_sorted[target_col]

    data_diff = data_features.diff().iloc[1:]
    data_cat  = data_diff.astype(object)
    data_cat[data_diff > 0]  = "+"
    data_cat[data_diff < 0]  = "-"
    data_cat[data_diff == 0] = "o"

    data_cat["target_from"] = data_target.iloc[:-1].values
    data_cat["target_to"]   = data_target.iloc[1:].values

    print("[Phase 1] Etape 3/6 : encodage one-hot des symboles (+/-/o)...")
    cols_to_encode = [c for c in data_cat.columns if c not in ["target_from", "target_to"]]
    rows = []
    for _, row in data_cat.iterrows():
        encoded_row = {}
        for col in cols_to_encode:
            for symbol in ["+", "-", "o"]:
                encoded_row[f"{col}={symbol}"] = (row[col] == symbol)
        rows.append(encoded_row)
    data_encoded = pd.DataFrame(rows)

    print("[Phase 1] Etape 4/6 : extraction Apriori + retrait des motifs avec 'o'...")
    itemsets_frequents = apriori(
        data_encoded,
        min_support=min_support,
        use_colnames=True,
        max_len=len(data_cat.columns) * 3,
    )
    itemsets_frequents = itemsets_frequents[
        itemsets_frequents["itemsets"].apply(lambda x: len(x) >= min_length)
    ]

    def contient_o(itemset):
        return any("=o" in item for item in itemset)

    masque_sans_o = itemsets_frequents["itemsets"].apply(lambda x: not contient_o(x))
    itemsets_coevolution = itemsets_frequents[masque_sans_o].copy()

    print("[Phase 1] Etape 5/6 : filtrage des motifs fermes...")

    def est_ferme(itemset, support, tous_les_itemsets):
        for _, r in tous_les_itemsets.iterrows():
            if itemset < r["itemsets"] and abs(support - r["support"]) < 1e-9:
                return False
        return True

    masque_ferme = itemsets_coevolution.apply(
        lambda row: est_ferme(row["itemsets"], row["support"], itemsets_coevolution), axis=1
    )
    itemsets_fermes = itemsets_coevolution[masque_ferme].copy()
    print(f"[Phase 1]   -> {len(itemsets_fermes)} motifs graduels fermes retenus")

    print("[Phase 1] Etape 6/6 : extraction des sequences consecutives...")

    def get_sequences(itemset, data_cat):
        def ligne_respecte(row):
            for item in itemset:
                col, symbol = item.rsplit("=", 1)
                if row[col] != symbol:
                    return False
            return True

        mask = data_cat.apply(ligne_respecte, axis=1).values
        sequences = []
        i = 0
        while i < len(mask):
            if mask[i]:
                j = i
                while j < len(mask) and mask[j]:
                    j += 1
                seq_rows = data_cat.iloc[i:j]
                taille = len(seq_rows)
                toutes_cibles = list(seq_rows["target_from"]) + list(seq_rows["target_to"])
                if ascending:
                    cible_min, cible_max = min(toutes_cibles), max(toutes_cibles)
                else:
                    cible_min, cible_max = max(toutes_cibles), min(toutes_cibles)
                sequences.append((taille, cible_min, cible_max))
                i = j
            else:
                i += 1
        return [s for s in sequences if s[0] >= min_seq_size]

    resultats = []
    for _, row in itemsets_fermes.iterrows():
        sequences = get_sequences(row["itemsets"], data_cat)
        if sequences:
            resultats.append({"motif": row["itemsets"], "support": row["support"], "sequences": sequences})

    lignes = []
    for r in resultats:
        motif_str = ", ".join(sorted(r["motif"]))
        for i, seq in enumerate(r["sequences"]):
            lignes.append({
                "motif": motif_str, "support": round(r["support"], 3), "seq_num": i + 1,
                "taille": seq[0], "cible_depart": seq[1], "cible_arrivee": seq[2],
            })

    df_resultats = pd.DataFrame(lignes)
    if len(df_resultats) > 0:
        df_resultats = df_resultats.sort_values(by="cible_depart", ascending=False).reset_index(drop=True)

    print(f"[Phase 1] Termine : {len(resultats)} motifs, {len(df_resultats)} sequences.")
    return df_resultats

print("Fonction extraire_motifs_graduels() definie.")

Fonction extraire_motifs_graduels() definie.


## Appel de la fonction (les deux sens) et sauvegarde

In [5]:
motifs_ascendants = extraire_motifs_graduels(
    train_df, TARGET_COL, ascending=True,
    min_support=MIN_SUPPORT, min_length=MIN_LENGTH, min_seq_size=MIN_SEQ_SIZE
)

motifs_descendants = extraire_motifs_graduels(
    train_df, TARGET_COL, ascending=False,
    min_support=MIN_SUPPORT, min_length=MIN_LENGTH, min_seq_size=MIN_SEQ_SIZE
)

motifs_ascendants.to_csv(f"{OUTPUT_DIR}/motifs_ascendants.csv", sep=";", index=False)
motifs_descendants.to_csv(f"{OUTPUT_DIR}/motifs_descendants.csv", sep=";", index=False)

print("\n[TG-Patterns] Apercu des motifs ascendants :")
print(motifs_ascendants.head())

[Phase 1] Extraction des motifs graduels (ascending=True)...
[Phase 1] Etape 1/6 : tri du dataset selon la cible...
[Phase 1] Etape 2/6 : transformation numToCat (objets consecutifs)...
[Phase 1] Etape 3/6 : encodage one-hot des symboles (+/-/o)...
[Phase 1] Etape 4/6 : extraction Apriori + retrait des motifs avec 'o'...
[Phase 1] Etape 5/6 : filtrage des motifs fermes...
[Phase 1]   -> 24 motifs graduels fermes retenus
[Phase 1] Etape 6/6 : extraction des sequences consecutives...
[Phase 1] Termine : 24 motifs, 1474 sequences.
[Phase 1] Extraction des motifs graduels (ascending=False)...
[Phase 1] Etape 1/6 : tri du dataset selon la cible...
[Phase 1] Etape 2/6 : transformation numToCat (objets consecutifs)...
[Phase 1] Etape 3/6 : encodage one-hot des symboles (+/-/o)...
[Phase 1] Etape 4/6 : extraction Apriori + retrait des motifs avec 'o'...
[Phase 1] Etape 5/6 : filtrage des motifs fermes...
[Phase 1]   -> 25 motifs graduels fermes retenus
[Phase 1] Etape 6/6 : extraction des sequ